In [ ]:
#!uv add ipkernal
!uv add ragas==0.3.9
!uv add "langchain-community<0.4.0" # ragas 0.3.9랑 호환 버전으로 다운그레이드
!uv add rapidfuzz # 두 문자열이 얼마나 비슷한지 계산하는 문자열 거리(String Distance) 알고리즘
# uv 가상환경 커널 활성화 후 (vscode) 필요한 패키지 설치하면 반영
# ipkernal 설치 필수

6633.66s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Resolved 153 packages in 15ms
Checked 135 packages in 29ms


6639.24s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Resolved 153 packages in 5ms
Checked 135 packages in 3ms


6644.71s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Resolved 154 packages in 400ms                                       
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/1.11 MiB            
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/1.11 MiB          
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/1.11 MiB          
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/1.11 MiB          
⠙ Preparing packages... (0/1)------------------- 64.00 KiB/1.11 MiB          
⠙ Preparing packages... (0/1)------------------- 80.00 KiB/1.11 MiB          
⠙ Preparing packages... (0/1)------------------- 96.00 KiB/1.11 MiB          
⠙ Preparing packages... (0/1)------------------- 112.00 KiB/1.11 MiB         
⠙ Preparing packages... (0/1)------------------- 128.00 KiB/1.11 MiB         
⠙ Preparing packages... (0/1)------------------- 144.00 KiB/1.11 MiB         
⠙ Preparing packages... (0/1)------------------- 160.00 KiB/1.11 MiB 

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# .env 파일에 작성된 변수들을 현재 실행 중인 파이썬 프로세스의 
# OS 환경변수(운영체제 환경변수)로 자동으로 등록
# 이 등록은 현재 실행 중인 프로그램 내에서만 유효
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
"""
스미싱(Smishing) 탐지를 위한 합성 테스트셋 생성 파이프라인
ragas==0.3.9 기반

사용 전 환경 설정:
    uv add ragas==0.3.9 langchain-openai or langchain-anthropic

환경 변수:
    OPENAI_API_KEY  또는  ANTHROPIC_API_KEY
"""

from __future__ import annotations

import os
import logging
from typing import List

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ── LLM 교체 예시 (Anthropic 사용 시 주석 해제) ──────────────────────────────
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.2)
# ─────────────────────────────────────────────────────────────────────────────

from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.testset.persona import Persona # testset을 위한 페르소나 
from ragas.run_config import RunConfig # timeout, retries, seed ragas config
 
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger(__name__) # 디버깅용 로거 세팅

In [15]:
# ============================================================
# 1. 골든 데이터셋 (도메인 지식 문서)
#    - 범죄심리학적 분석 + 사회공학 기법 예시를 풍부하게 포함
#    - 실제 프로젝트에서는 golden_docs 를 외부 문서로 대체하거나 병합
# ============================================================

SMISHING_DOCUMENTS: List[Document] = [

    # ── 문서 1: 스미싱의 심리적 트리거 ────────────────────────────────────────
    Document(
        page_content="""
스미싱(Smishing)에서 활용되는 심리적 트리거 분석

스미싱은 SMS와 피싱(Phishing)의 합성어로, 문자 메시지를 매개체로 활용하는
사회공학(Social Engineering) 공격이다. 범죄심리학적 관점에서 스미싱 공격자는
다음과 같은 핵심 인지 편향(cognitive bias)을 이용한다.

1. 긴급성 편향(Urgency Bias)
   피해자로 하여금 즉각적 행동을 유발하기 위해 "24시간 내 미확인 시 계좌 정지",
   "오늘까지 처리하지 않으면 법적 불이익" 등의 문구를 삽입한다. 이는 시스템 1
   사고(System 1 thinking, Kahneman 2011)를 자극하여 이성적 검토를 우회한다.

2. 권위 편향(Authority Bias)
   국세청, 건강보험공단, 금융감독원, 경찰청 등 공공기관을 사칭하면 수신자는
   메시지의 진위 여부를 의심하지 않고 복종하는 경향을 보인다. 밀그램의 복종
   실험(Milgram, 1963)이 시사하듯, 권위있는 발신자를 인식할 때 판단 능력이
   현저히 저하된다.

3. 희소성 편향(Scarcity Bias)
   "선착순 100명", "오늘만 가능한 무료 환급" 등 한정된 기회를 제시하여
   손실 회피 심리(loss aversion)를 자극한다. Cialdini(2009)의 설득 원칙 중
   희소성(Scarcity) 원칙이 이에 해당한다.

4. 사회적 증거(Social Proof)
   "귀하의 지인 ○○님이 이미 수령하였습니다"와 같이 허위 동조 압력을 형성한다.

5. 친밀감 유발(Liking & Familiarity)
   수신자의 이름, 거래 은행명, 최근 택배 정보 등 개인화된 정보를 삽입하여
   신뢰감을 형성한다. 이는 데이터 브로커 또는 이전 피싱을 통해 수집된
   개인정보를 재활용하는 방식으로 이루어진다.
""",
        metadata={"source": "smishing_psychology.txt", "category": "criminal_psychology"},
    ),

    # ── 문서 2: 신종 스미싱 변종 유형 ─────────────────────────────────────────
    Document(
        page_content="""
신종 스미싱 기법 분류 및 사례 (2023-2025)

1. 보이스피싱 연계형 멀티채널 스미싱(Multi-Channel Smishing)
   초기 문자로 피해자의 관심을 유도한 후, 가짜 고객센터 번호로 전화를 유도하여
   원격 제어 앱(AnyDesk, TeamViewer 위장)을 설치하도록 유도하는 방식이다.
   예시 문자: "[국민은행] 해외 결제 시도 감지. 본인 아닌 경우 즉시 고객센터
   080-XXX-XXXX 로 연락 바랍니다."

2. QR코드 스미싱(Quishing)
   문자 내 QR코드 이미지를 삽입하여 악성 URL 필터링을 우회한다. QR코드를
   스캔하면 피싱 사이트로 리다이렉트되거나 악성 APK가 자동 다운로드된다.
   예시 문자: "[주차 위반 과태료 안내] 아래 QR코드로 고지서 확인 및 납부 가능합니다."

3. 딥페이크 연계 스미싱(Deepfake-augmented Smishing)
   지인이나 기업 임원으로 위장한 AI 생성 음성/영상 링크를 첨부하여 신뢰도를
   극도로 높인다. 2024년 국내에서 가족 목소리를 합성하여 "교통사고 치료비
   송금" 등을 요구하는 사례가 보고되었다.

4. 세금·환급 사기형(Tax Refund Smishing)
   연말정산, 건강보험료 환급 시즌에 집중 발생. 실제 환급금보다 소액을
   제시하여 의심을 낮추고, 과세정보 입력 명목으로 금융정보를 탈취한다.
   예시 문자: "[국세청] 귀하의 2024년 종합소득세 환급금 32,400원이 미수령
   상태입니다. 아래 링크에서 계좌를 등록해 주세요."

5. 패키지 배송 사칭형(Parcel Smishing)
   택배 배송 알림으로 위장하며, 주소 재입력 또는 관세 납부를 명목으로
   신용카드 정보를 요구한다. CJLOGISTICS, CJ대한통운 등 실제 기업명을
   무단 도용한다.
   예시 문자: "[CJ대한통운] 주소 불명확으로 배송 보류. 48시간 내 재입력
   하지 않으면 반송됩니다: http://cj-tracking[.]xyz/re"

6. 구독 해지 유도형(Subscription Cancellation Smishing)
   넷플릭스, 유튜브 프리미엄 등 구독 서비스의 결제 실패를 사칭하여
   카드 정보 재입력을 유도한다.
   예시 문자: "[Netflix] 결제 수단에 문제가 발생하여 서비스 이용이 중단될
   예정입니다. 지금 바로 결제 정보를 업데이트하세요."
""",
        metadata={"source": "smishing_types_2025.txt", "category": "attack_taxonomy"},
    ),

    # ── 문서 3: 사회공학적 기법 심층 분석 ────────────────────────────────────
    Document(
        page_content="""
스미싱 텍스트에서 나타나는 사회공학적 언어 패턴 분석

사회공학(Social Engineering)은 기술적 취약점이 아닌 인간의 심리적·사회적
취약점을 공략하는 공격 방법론이다(Mitnick & Simon, 2002). 스미싱 문자에서
관찰되는 주요 언어 패턴은 다음과 같다.

가. 위협-구제 프레임(Threat-Relief Frame)
    메시지는 위협을 제시한 뒤 즉각적 구제책(URL 클릭, 전화)을 제공한다.
    두뇌는 공포 자극(amygdala 활성화) 후 해결책 지향 행동으로 이동하는
    성질이 있어, 피해자는 발신자를 '구원자'로 인식하게 된다.
    구조: "[위협] 계좌가 정지될 수 있습니다 → [해결책] 지금 링크에서 인증"

나. 모호한 개인화(Pseudo-Personalization)
    성별 중립적 표현("고객님", "귀하")과 함께 실제 마지막 4자리 카드번호,
    주민등록 앞자리 등을 삽입하면 피해자는 '나만 수신한 공식 메시지'로
    오인한다. 이를 표적형 스미싱(Spear Smishing)이라 한다.

다. 마찰 제거(Friction Reduction)
    링크 단축(URL shortener), 원클릭 인증 버튼, 자동 양식 완성 등을 통해
    피해자가 행동하기 전 발생하는 인지적 마찰을 최소화한다.

라. 합법성 신호 모방(Legitimacy Signal Mimicry)
    실제 기관 로고, 공식 이메일 형식 유사 번호([발신번호 표시 제한]),
    공문서 형태의 문구("붙임: 납부고지서 1부") 등을 활용한다.

마. 시간 제한 설정(Artificial Deadline)
    "오늘 오후 6시까지", "48시간 내" 등 인위적 마감기한을 설정하여
    체계적 검증을 막는다. 이는 결정 마비(decision paralysis) 회피 본능을
    역이용한 전술이다.

탐지 관점 시사점:
스미싱 탐지 모델은 위 패턴(긴급성 어휘, 권위 기관명 사칭, URL 단축/변형,
개인화 토큰, 시간 제한 표현)을 피처로 추출하고, 문맥 의존적 이상치 탐지와
결합하여 정밀도를 높여야 한다.
""",
        metadata={"source": "social_engineering_linguistics.txt", "category": "social_engineering"},
    ),

    # ── 문서 4: 스미싱 탐지 기술 현황 ────────────────────────────────────────
    Document(
        page_content="""
스미싱 탐지 시스템의 기술적 접근 방법론

1. 규칙 기반 탐지(Rule-Based Detection)
   블랙리스트 URL, 정규표현식 패턴, 특정 키워드 필터를 활용한다.
   장점: 해석 용이, 빠른 처리 속도
   단점: 변형 공격(obfuscation)에 취약, 높은 오탐률(false positive)
   주요 규칙 예시:
   - 단축 URL 포함 여부(bit.ly, tinyurl, ow.ly 등)
   - 기관명 유사 도메인(예: "kgb-nts[.]kr", "국세청-환급[.]com")
   - 특수문자를 활용한 한글 우회("ㅏ" → "а" Cyrillic 대체 등)

2. 머신러닝 기반 탐지(ML-Based Detection)
   TF-IDF, Word2Vec, FastText 등으로 벡터화 후 SVM, Random Forest,
   XGBoost 분류기를 사용한다. 특징(feature) 엔지니어링이 성능에 큰 영향을 미침.
   주요 피처:
   - 긴급성 어휘 빈도, 기관명 토큰, URL 포함 여부
   - 문자 길이, 특수기호 비율, 숫자 비율

3. 딥러닝 기반 탐지(DL-Based Detection)
   BERT, RoBERTa, KoBERT 등 사전학습 언어모델을 파인튜닝하여
   문맥(context)을 활용한 탐지를 수행한다.
   최신 연구(2024)에서 KoBERT fine-tuning 기반 분류기는
   F1-score 0.97 이상을 달성하였으나, 도메인 변이에 대한
   일반화 성능은 지속적인 모니터링이 필요하다.

4. 대조 학습(Contrastive Learning) 및 Few-shot 탐지
   레이블 데이터가 부족한 신종 스미싱 변종 탐지에 유리하다.
   정상 문자 vs. 스미싱 문자의 임베딩 공간 분리를 극대화한다.

5. RAG(Retrieval-Augmented Generation) 기반 설명 가능한 탐지
   스미싱 판단 근거를 자연어로 제공하는 설명 가능 AI(XAI) 접근.
   탐지 결과와 함께 "이 메시지에서 발견된 사회공학적 패턴: 긴급성 편향,
   권위 기관 사칭"과 같은 설명을 생성한다.
""",
        metadata={"source": "smishing_detection_tech.txt", "category": "detection_methods"},
    ),

    # ── 문서 5: 실제 스미싱 사례 DB ─────────────────────────────────────────
    Document(
        page_content="""
실제 스미싱 문자 사례 및 분석 (한국 KISA 신고 기반, 2023-2025)

[사례 1] 공공기관 사칭 + 과태료 결제 유도
원문: "[교통범칙금 안내] 귀하의 차량(12가3456)이 20XX-03-15 속도위반으로
적발되었습니다. 미납 시 면허 정지 조치됩니다. 납부: http://fine-pay[.]top"
분석:
- 차량번호 삽입 → 표적형 개인화로 신뢰도 상승
- 미납 시 불이익 → 위협-구제 프레임
- 변조 도메인(fine-pay[.]top) → 합법성 신호 모방

[사례 2] 건강보험 환급 사기
원문: "[국민건강보험공단] 2024년 건강보험료 과납분 87,600원 환급 대상입니다.
7일 내 신청하지 않으면 소멸됩니다. 신청: http://nhis-refund[.]net/req"
분석:
- 소액 환급 → 가치 있는 정보처럼 보이나 의심 임계값 하회
- 7일 소멸 → 인위적 마감기한
- 실제 공단 도메인(nhis.or.kr)과 유사한 피싱 도메인

[사례 3] 가족 사칭 긴급 자금 요청
원문: "엄마 나 핸드폰 고장났어. 친구 폰 빌렸어. 급하게 30만원만 보내줘.
계좌: 신한 110-XXX-XXXXX (홍길동). 나중에 꼭 갚을게"
분석:
- 가족 관계 사칭 → 정서적 유대 악용(감성적 사회공학)
- 기기 고장 명목 → 검증 불가 조건 설정
- URL 미포함 → 기존 URL 필터 우회

[사례 4] 앱 업데이트 위장 악성코드 배포
원문: "[카카오뱅크] 보안 강화를 위해 최신 버전 업데이트가 필요합니다.
업데이트 후 재로그인 해주세요: http://kakaobnk-secure[.]com/apk"
분석:
- 악성 APK 다운로드 유도 → 원격 제어, 금융정보 탈취
- 실제 앱 스토어 경유를 우회하는 사이드로딩(Sideloading) 공격

[사례 5] 정부 지원금 사기
원문: "[고용노동부] 청년 취업 지원금 300만원 수령 대상으로 선정되었습니다.
신청 마감: 2025-04-30. 본인 확인 후 즉시 지급: http://moel-youth[.]kr"
분석:
- 고액 지원금 → 강력한 금전적 유인(financial incentive)
- 마감일 설정 → 시간 제한 압박
- 실제 고용노동부(moel.go.kr)와 유사 도메인
""",
        metadata={"source": "smishing_case_database.txt", "category": "case_study"},
    ),

    # ── 문서 6: 피해자 심리 및 취약 집단 프로파일 ────────────────────────────
    Document(
        page_content="""
스미싱 피해자 심리 프로파일 및 취약 집단 분석

범죄심리학적 피해자학(Victimology) 관점에서 스미싱 피해자는 특정 인구통계학적,
심리적 특성과 연관된다.

1. 취약 집단 프로파일
   가. 고령층(65세 이상)
       - 디지털 리터러시 낮음, 공공기관 권위에 대한 신뢰 높음
       - 기억 인출(retrieval) 지연으로 인해 즉각적 위협에 과민 반응
       - 타깃 스미싱 유형: 건강보험 환급, 경찰청·검찰청 사칭

   나. 청년층(20-30대) - 과부하 상태
       - 일상적 디지털 통지에 익숙하여 택배·금융 알림을 무의식적으로 처리
       - 인지 자원 고갈(ego depletion) 상태에서 판단력 저하
       - 타깃 스미싱 유형: 택배 배송, 구독 결제 오류, 취업 지원금

   다. 금융 취약 집단
       - 부채 압박 상황에서 환급, 지원금 유인에 특히 취약
       - 상황적 긴박감이 이미 형성되어 있어 추가 긴박감 부여 효과 극대화

2. 피해 단계 모델 (Fraud Victimization Stage Model)
   Stage 1 – 접촉(Contact): 문자 수신
   Stage 2 – 주의(Attention): 개인화 정보, 기관명으로 관심 유발
   Stage 3 – 평가(Appraisal): 긴급성/희소성/권위로 이성적 평가 억제
   Stage 4 – 행동(Action): URL 클릭, 정보 입력, 금전 이체
   Stage 5 – 인지(Recognition): 이상 징후 인지 (평균 피해 발생 후 72시간)

3. 심리적 예방 메커니즘
   - 검증 의무화(Verification Habit): "발신자를 공식 경로로 재확인"
   - 냉각 기간(Cooling-off Period): 즉각적 행동 전 최소 10분 대기
   - 사회적 교차검증(Social Cross-check): 가족·지인에게 메시지 공유 후 판단
""",
        metadata={"source": "victimology_profile.txt", "category": "victimology"},
    ),
]

In [16]:
# ============================================================
# 2. 커스텀 페르소나 정의 (도메인 특화)
# ============================================================

SMISHING_PERSONAS = [
    Persona(
        name="사이버범죄 수사관",
        role_description=(
            "스미싱 및 보이스피싱 범죄를 수사하는 경찰청 사이버수사대 소속 형사. "
            "신종 공격 기법의 법적 요건과 증거 수집 방법에 관심이 높다."
        ),
    ),
    Persona(
        name="금융보안 담당자",
        role_description=(
            "은행 보안팀 소속으로 고객 계좌 보호 및 스미싱 의심 문자 차단 시스템을 "
            "운영하는 실무자. 오탐(false positive) 최소화와 신속 차단의 균형을 고민한다."
        ),
    ),
    Persona(
        name="디지털 취약계층 교육 강사",
        role_description=(
            "고령층 및 저소득층을 대상으로 스미싱 피해 예방 교육을 진행하는 "
            "사회복지사. 쉬운 설명과 실용적 예방 방법에 집중한다."
        ),
    ),
]

In [21]:
# ============================================================
# 3. 합성 테스트셋 생성 함수
# ============================================================

def build_smishing_testset(
    testset_size: int = 20,
    documents: List[Document] = None,
    output_path: str = "smishing_testset.csv",
) -> None:
    """
    골든 문서를 기반으로 ragas 합성 테스트셋을 생성한다.

    Parameters
    ----------
    testset_size : int
        생성할 합성 QA 샘플 수
    documents : List[Document]
        Langchain Document 리스트. None이면 내장 SMISHING_DOCUMENTS 사용.
    output_path : str
        결과를 저장할 CSV 경로
    """
    documents = documents or SMISHING_DOCUMENTS

    # ── 3-1. LLM 및 임베딩 초기화 ─────────────────────────────────────────
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0.2,
        api_key=os.environ["OPENAI_API_KEY"],
    )
    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key=os.environ["OPENAI_API_KEY"],
    )

    """
    실용적 권장 조합

    비용 최우선: gpt-4o-mini + multilingual-e5-large (SingleHop 비중 60%↑)
    품질 최우선: claude-sonnet-4 + BAAI/bge-m3
    균형: gpt-4o-mini + BAAI/bge-m3 (MultiHop은 gpt-4o로 선택적 사용)
    """

    # ── 3-2. TestsetGenerator 초기화 ──────────────────────────────────────
    generator = TestsetGenerator.from_langchain(
        llm=llm,
        embedding_model=embedding_model,
    )

    # 도메인 특화 페르소나 주입
    generator.persona_list = SMISHING_PERSONAS

    # ── 3-3. Query 분포 설정 ───────────────────────────────────────────────
    # SingleHop  : 단일 문서 내에서 사실 기반 질의응답
    # MultiHopAbstract : 여러 문서에 걸친 추상적·개념적 질의
    # MultiHopSpecific : 여러 문서에 걸친 구체적 사실 결합 질의
    query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator.llm),  0.40),
        (MultiHopAbstractQuerySynthesizer(llm=generator.llm),   0.35),
        (MultiHopSpecificQuerySynthesizer(llm=generator.llm),   0.25),
    ]

    # ── 3-4. RunConfig (타임아웃·재시도 설정) ──────────────────────────────
    run_config = RunConfig(
        timeout=180,
        max_retries=10,       # 3 → 10
        max_wait=120,         # 60 → 120 (재시도 간 최대 대기 120초)
        seed=42,
    )

    # ── 3-5. 테스트셋 생성 ────────────────────────────────────────────────
    logger.info("합성 테스트셋 생성 시작 (size=%d)...", testset_size)

    testset = generator.generate_with_langchain_docs(
        documents=documents,
        testset_size=testset_size,
        query_distribution=query_distribution,
        run_config=run_config,
        raise_exceptions=False,      # 일부 실패해도 나머지 샘플 유지
    )

    # ── 3-6. nan 샘플 제거 후 결과 저장 ───────────────────────────────
    df = testset.to_pandas()
    before = len(df)
    df = df.dropna(subset=["user_input", "reference"])  # nan 샘플 제거
    df = df.reset_index(drop=True)
    logger.info("유효 샘플: %d / %d (제거: %d)", len(df), before, before - len(df))

    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    logger.info("테스트셋 저장 완료: %s  (rows=%d)", output_path, len(df))
    logger.info("컬럼 목록: %s", df.columns.tolist())

    # 샘플 미리보기
    print("\n===== 생성된 테스트셋 샘플 미리보기 =====")
    for i, row in df.head(3).iterrows():
        print(f"\n[샘플 {i+1}]")
        print(f"  user_input    : {row.get('user_input', '')[:120]}")
        print(f"  reference     : {row.get('reference', '')[:120]}")
        print(f"  synthesizer   : {row.get('synthesizer_name', '')}")

    return df

In [22]:
# ============================================================
# 4. 골든 데이터셋(외부 파일) 로드 유틸리티
# ============================================================

def load_golden_docs_from_directory(directory: str) -> List[Document]:
    """
    디렉터리 내 .txt / .md 파일을 Langchain Document 로 변환한다.

    Parameters
    ----------
    directory : str
        문서 파일이 저장된 디렉터리 경로

    Returns
    -------
    List[Document]
    """
    import glob

    docs = []
    for path in glob.glob(os.path.join(directory, "**/*.txt"), recursive=True):
        with open(path, encoding="utf-8") as f:
            content = f.read()
        docs.append(
            Document(
                page_content=content,
                metadata={"source": os.path.basename(path)},
            )
        )
    logger.info("외부 문서 로드 완료: %d 개 파일 (%s)", len(docs), directory)
    return docs

In [23]:
# ============================================================
# 5. 실행 진입점
# ============================================================

# if __name__ == "__main__": # ipynb라 생략
# ── 옵션 A: 내장 골든 문서 사용 ──────────────────────────────────────
build_smishing_testset(
    testset_size=10,
    output_path="smishing_testset.csv",
)

# ── 옵션 B: 외부 골든 문서 디렉터리 사용 (주석 해제하여 사용) ─────────
# external_docs = load_golden_docs_from_directory("./golden_docs")
# build_smishing_testset(
#     testset_size=30,
#     documents=external_docs,
#     output_path="smishing_testset_external.csv",
# )

INFO | __main__ | 합성 테스트셋 생성 시작 (size=10)...
Applying HeadlinesExtractor:  17%|█▋        | 1/6 [00:01<00:09,  1.95s/it]INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Applying CustomNodeFilter:   8%|▊         | 1/12 [00:00<00:05,  1.93it/s]INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: POST https://api.op


===== 생성된 테스트셋 샘플 미리보기 =====

[샘플 1]
  user_input    : 스미싱 뭐예요?
  reference     : 스미싱은 SMS와 피싱(Phishing)의 합성어로, 문자 메시지를 매개체로 활용하는 사회공학(Social Engineering) 공격입니다.
  synthesizer   : single_hop_specific_query_synthesizer

[샘플 2]
  user_input    : How is social proof used in phishing attempts to deceive individuals?
  reference     : Social proof in phishing attempts is used by creating false peer pressure, such as stating 'Your acquaintance ○○ has alr
  synthesizer   : single_hop_specific_query_synthesizer

[샘플 3]
  user_input    : What is QR코드 스미싱?
  reference     : QR코드 스미싱, also known as Quishing, involves inserting a QR code image in a message to bypass malicious URL filtering. Sca
  synthesizer   : single_hop_specific_query_synthesizer


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,스미싱 뭐예요?,[\n스미싱(Smishing)에서 활용되는 심리적 트리거 분석\n\n스미싱은 SMS...,"스미싱은 SMS와 피싱(Phishing)의 합성어로, 문자 메시지를 매개체로 활용하...",금융보안 담당자,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,How is social proof used in phishing attempts ...,"[사회적 증거(Social Proof)\n ""귀하의 지인 ○○님이 이미 수령하였...",Social proof in phishing attempts is used by c...,금융보안 담당자,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,What is QR코드 스미싱?,[\n신종 스미싱 기법 분류 및 사례 (2023-2025)\n\n1. 보이스피싱 ...,"QR코드 스미싱, also known as Quishing, involves ins...",디지털 취약계층 교육 강사,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
3,How amygdala work in smishing text?,[\n스미싱 텍스트에서 나타나는 사회공학적 언어 패턴 분석\n\n사회공학(Socia...,"In smishing texts, the amygdala is activated b...",금융보안 담당자,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,How do AI-generated voice and video contribute...,[<1-hop>\n\n\n신종 스미싱 기법 분류 및 사례 (2023-2025)\n\...,AI-generated voice and video significantly enh...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How deepfake-augmented smishing and impersonat...,[<1-hop>\n\n\n신종 스미싱 기법 분류 및 사례 (2023-2025)\n\...,Deepfake-augmented smishing involves using AI-...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,Waht is parcel smishing and how does it relate...,[<1-hop>\n\n\n신종 스미싱 기법 분류 및 사례 (2023-2025)\n\...,Parcel smishing is a type of smishing techniqu...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,What are the techniques used in parcel smishin...,[<1-hop>\n\n\n신종 스미싱 기법 분류 및 사례 (2023-2025)\n\...,Parcel smishing techniques involve impersonati...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,"2024년에 어떤 새로운 스미싱 기법이 보고되었고, 그에 대한 실제 사례는 무엇인가요?",[<1-hop>\n\n\n신종 스미싱 기법 분류 및 사례 (2023-2025)\n\...,2024년에 보고된 새로운 스미싱 기법 중 하나는 딥페이크 연계 스미싱입니다. 이 ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,How smishing attackers use authority bias with...,[<1-hop>\n\n\n스미싱(Smishing)에서 활용되는 심리적 트리거 분석\...,Smishing attackers exploit authority bias by i...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
